# Part 2 — 추가 데이터셋 · 비교 모델

위에서 찍은 사진(UC Merced, NWPU-RESISC45, AID)과 앞에서 찍은 사진(SUN-AID: SUN397에서 AID와 같은 개념의 장면)에서, SAM 3 · CLIP · RemoteCLIP에 같은 모듈(OCT, MPV, PPR, PSC)과 비교 기준(TransLP, TransCLIP-lite, 선형 분류기)을 적용해요.

- 런타임: GPU(L4 권장). `HF_TOKEN`을 Colab Secrets에 등록하고 `facebook/sam3` 접근 승인이 필요해요.
- 점수 캐시는 `WORK/bench/`에 쌓이고, 셀이 끊겨도 다시 실행하면 이어서 돼요.
- AID의 SAM 3 점수는 Part 1(`notebooks/colab_demo.ipynb`) 캐시를 다시 써요.


In [ ]:
# [1] 환경 세팅
import os
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive, userdata
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/SAM3_PGLC'
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['HF_DATASETS_DISABLE_PROGRESS_BARS'] = '1'
if not os.path.exists('/content/sam3'):
    !git clone -q https://github.com/facebookresearch/sam3 /content/sam3
!pip install -q --no-deps -e /content/sam3
!pip install -q timm ftfy iopath datasets open_clip_torch
# 이 저장소 (비공개 저장소면 토큰이 들어간 URL이나 Drive에 올린 사본을 쓰세요)
if not os.path.exists('/content/CLEAR_SAM3'):
    !git clone -q https://github.com/ing0333/CLEAR_SAM3 /content/CLEAR_SAM3

In [ ]:
# [2] CLIP · RemoteCLIP: 데이터 준비(처음 한 번) + 4개 데이터셋 — 약 40분 (SUN397 7GB 스트리밍이 가장 오래 걸려요)
!cd /content/CLEAR_SAM3 && python -u experiments/multi_benchmark.py --work_dir {WORK} --datasets ucm,sun_aid,resisc45,aid --models openai,remoteclip 2>&1 | grep --line-buffered -v -i -E "warn|futurewarning"

In [ ]:
# [3] SAM 3: AID(Part 1 캐시 재사용) + UC Merced — 약 40분
!cd /content/CLEAR_SAM3 && python -u experiments/multi_benchmark.py --work_dir {WORK} --datasets aid,ucm --models sam3 2>&1 | grep --line-buffered -v -i -E "warn|futurewarning"

In [ ]:
# [4] SAM 3: SUN-AID (앞에서 찍은 사진) — 약 50분
!cd /content/CLEAR_SAM3 && python -u experiments/multi_benchmark.py --work_dir {WORK} --datasets sun_aid --models sam3 2>&1 | grep --line-buffered -v -i -E "warn|futurewarning"

In [ ]:
# [5] SAM 3: NWPU-RESISC45 (표현 222개) — 약 2.5시간
!cd /content/CLEAR_SAM3 && python -u experiments/multi_benchmark.py --work_dir {WORK} --datasets resisc45 --models sam3 2>&1 | grep --line-buffered -v -i -E "warn|futurewarning"

In [ ]:
# [6] 결과 표 (README와 같은 형식)
!cd /content/CLEAR_SAM3 && python experiments/make_report.py {WORK}/bench/multi_benchmark.json